# Chapter 42: Time-Series Analysis and Forecasting

Synthetic weekly NRG demand demonstrates seasonal baselines, rolling origins, and decision-aligned evaluation.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0,str(Path.cwd().parents[1]/'src'))
from datasciencebook.forecasting import naive_forecast,seasonal_naive_forecast,drift_forecast,mae,rmse,mase,rolling_origins
print('Imports ready.')


Imports ready.


In [ ]:
rng=np.random.default_rng(42);weeks=np.arange(156)
demand=520+1.15*weeks+75*np.sin(2*np.pi*weeks/52)+rng.normal(0,18,len(weeks))
train=demand[:130];test=demand[130:];h=len(test)
print(f'Train weeks: {len(train)}; test weeks: {len(test)}; horizon: {h}')


Train weeks: 130; test weeks: 26; horizon: 26


In [ ]:
forecasts={'naive':naive_forecast(train,h),'seasonal_naive':seasonal_naive_forecast(train,h,52),'drift':drift_forecast(train,h)}
for name,p in forecasts.items():print(f'{name}: MAE={mae(test,p):.1f} RMSE={rmse(test,p):.1f} MASE={mase(test,p,train,52):.2f}')


naive: MAE=33.2 RMSE=40.3 MASE=0.59
seasonal_naive: MAE=61.8 RMSE=66.4 MASE=1.09
drift: MAE=43.2 RMSE=50.4 MASE=0.76


In [ ]:
origins=rolling_origins(130,78,13,13);scores=[]
for _,end,start,stop in origins:
 p=seasonal_naive_forecast(demand[:end],stop-start,52);scores.append(mae(demand[start:stop],p))
print('Rolling-origin MAE:',', '.join(f'{v:.1f}' for v in scores));print(f'Mean={np.mean(scores):.1f}; range={min(scores):.1f}-{max(scores):.1f}')


Rolling-origin MAE: 56.6, 46.1, 55.9, 64.8
Mean=55.9; range=46.1-64.8


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].plot(weeks,demand,label='actual');axes[0].axvline(129,color='black',ls='--');axes[0].plot(weeks[130:],forecasts['seasonal_naive'],label='seasonal naive');axes[0].legend();axes[0].set(xlabel='Week',ylabel='Cases',title='Chronological holdout')
errors=test-forecasts['seasonal_naive'];axes[1].axhline(0,color='black');axes[1].plot(weeks[130:],errors,'o-');axes[1].set(xlabel='Week',ylabel='Forecast error',title='Residual sequence')
fig.tight_layout();plt.show()


## Interpretation

The plain naive forecast is strongest on this particular holdout, while seasonal-naive performance varies across earlier origins. A single split would hide that instability. Model selection should use repeated historical origins and the actual planning horizon.


In [ ]:
# Practice: add a promotion flag and test it using the same rolling origins.
